# Math500 Parse Repair & Rescoring

This notebook **does not rerun DeepSeek-R1 inference**.

It loads the previously saved benchmark results from Google Drive, repairs Math500 final-answer parsing, rescoring the saved generations, and writes corrected results to **new files** without overwriting the original.

### Why this notebook is needed

The original benchmark expected answers in the form:

```text
FINAL_ANSWER: ...
```

However, many Math500 generations use forms such as:

```text
**Final Answer:**
\[
\boxed{\dfrac{33}{100}}
\]
```

Those answers were previously marked as `missing_final_answer` even though the answer was actually present.

### Safety rule

This repair notebook only extracts answers from the **final-response region** or explicit final-answer markers. It does **not** guess an answer from an unfinished reasoning trace. Truncated generations that never reach a final answer remain failures.


## Install and import dependencies

In [1]:
!pip -q install sympy antlr4-python3-runtime==4.11

import ast
import json
import math
import re
from pathlib import Path

import numpy as np
import pandas as pd
import sympy as sp

pd.set_option("display.max_colwidth", 180)
pd.set_option("display.max_columns", 100)

print("Ready.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 144.2/144.2 kB 3.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
omegaconf 2.3.1 requires antlr4-python3-runtime==4.9.*, but you have antlr4-python3-runtime 4.11.0 which is incompatible.
Ready.


## Mount Google Drive and configure paths

In [2]:
from google.colab import drive
drive.mount("/content/drive")


INPUT_CSV = None

PROJECT_ROOT = Path(
    "/content/drive/MyDrive/DeepSeek_R1_7B_Quantization_Benchmark_v2"
)

# File names that the automatic finder will look for.
PREFERRED_INPUT_NAMES = [
    "all_results_with_overthinking.csv",
    "all_results.csv",
    "combined_results.csv",
]

OUTPUT_DIR = PROJECT_ROOT / "rescored" / "math500_parse_repair"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

CORRECTED_RESULTS_PATH = OUTPUT_DIR / "all_results_math500_rescored.csv"
MATH500_AUDIT_PATH = OUTPUT_DIR / "math500_rescore_audit.csv"
SUMMARY_PATH = OUTPUT_DIR / "math500_rescore_summary.csv"

print("Project root :", PROJECT_ROOT)
print("Output folder:", OUTPUT_DIR)

Mounted at /content/drive
Project root : /content/drive/MyDrive/DeepSeek_R1_7B_Quantization_Benchmark_v2
Output folder: /content/drive/MyDrive/DeepSeek_R1_7B_Quantization_Benchmark_v2/rescored/math500_parse_repair


## Find the saved results file in Drive

In [3]:
def find_results_file():
    if INPUT_CSV:
        p = Path(INPUT_CSV)
        if not p.exists():
            raise FileNotFoundError(f"Configured INPUT_CSV does not exist: {p}")
        return p

    # First search the expected project root.
    if PROJECT_ROOT.exists():
        for name in PREFERRED_INPUT_NAMES:
            matches = list(PROJECT_ROOT.rglob(name))
            if matches:
                # Prefer the newest file if duplicates exist.
                matches = sorted(
                    matches,
                    key=lambda p: p.stat().st_mtime,
                    reverse=True,
                )
                return matches[0]

    # Fallback: search MyDrive for only the preferred filenames.
    mydrive = Path("/content/drive/MyDrive")
    for name in PREFERRED_INPUT_NAMES:
        matches = list(mydrive.rglob(name))
        if matches:
            matches = sorted(
                matches,
                key=lambda p: p.stat().st_mtime,
                reverse=True,
            )
            return matches[0]

    raise FileNotFoundError(
        "Could not find a saved benchmark CSV. "
        "Set INPUT_CSV to the exact Drive path."
    )


SOURCE_PATH = find_results_file()

print("Using saved results:")
print(SOURCE_PATH)

Using saved results:
/content/drive/MyDrive/DeepSeek_R1_7B_Quantization_Benchmark_v2/all_results_with_overthinking.csv


## Load and validate the benchmark results

In [4]:
df = pd.read_csv(SOURCE_PATH)

required_columns = {
    "sample_id",
    "dataset",
    "precision",
    "gold_answer",
    "generated_text",
    "parse_status",
    "parse_success",
    "correct",
    "finish_reason",
    "was_truncated",
}

missing = sorted(required_columns - set(df.columns))
if missing:
    raise ValueError(f"Missing required columns: {missing}")

print(f"Rows loaded: {len(df):,}")
print(f"Columns    : {len(df.columns)}")
print()
print(df["dataset"].value_counts())
print()

math_df = df[df["dataset"].astype(str).str.lower().eq("math500")].copy()

print(f"Math500 rows: {len(math_df):,}")
display(
    math_df.groupby(["precision", "parse_status"])
    .size()
    .rename("rows")
    .reset_index()
)

Rows loaded: 1,656
Columns    : 43

dataset
date_understanding     300
math500                300
simpleqa_verified      300
strategyqa             300
logiqa2                300
misguided_attention    156
Name: count, dtype: int64

Math500 rows: 300


,precision,parse_status,rows
0,BF16,final_answer_found,1
1,BF16,missing_final_answer,49
2,Q3_K_M,final_answer_found,2
3,Q3_K_M,missing_final_answer,48
4,Q4_K_M,final_answer_found,3
5,Q4_K_M,missing_final_answer,47
6,Q5_K_M,missing_final_answer,50
7,Q6_K,final_answer_found,1
8,Q6_K,missing_final_answer,49
9,Q8_0,missing_final_answer,50


## Robust Math500 answer extraction

Extraction priority:

1. Exact `FINAL_ANSWER: ...`
2. Last balanced `\boxed{...}` in the final response
3. `Final Answer:` section
4. Last `\boxed{...}` in the full generation
5. Conservative natural-language final-answer phrase

The parser supports nested LaTeX such as:

```text
\boxed{\dfrac{\sqrt{3}}{3}}
```

It deliberately does **not** take the last number from an unfinished reasoning trace.


In [6]:
def extract_balanced_command(text, command=r"\boxed"):
    """
    Extract balanced {...} payloads following a LaTeX command such as \\boxed.
    Handles nested braces.
    """
    if not isinstance(text, str):
        return []

    hits = []
    start = 0

    while True:
        idx = text.find(command, start)
        if idx < 0:
            break

        j = idx + len(command)

        while j < len(text) and text[j].isspace():
            j += 1

        if j < len(text) and text[j] == "{":
            depth = 0
            found = False

            for k in range(j, len(text)):
                ch = text[k]
                escaped = k > 0 and text[k - 1] == "\\"

                if ch == "{" and not escaped:
                    depth += 1

                elif ch == "}" and not escaped:
                    depth -= 1

                    if depth == 0:
                        hits.append(text[j + 1:k])
                        start = k + 1
                        found = True
                        break

            if not found:
                start = j + 1

        else:
            # Handles rare forms like \\boxed 23
            m = re.match(r"([^\s$]+)", text[j:])
            if m:
                hits.append(m.group(1))
            start = j + 1

    return hits


def clean_candidate(s):
    if s is None:
        return ""

    s = str(s).strip()

    # Remove common display delimiters.
    s = s.replace(r"\[", "").replace(r"\]", "")
    s = s.strip("$").strip()

    # Remove markdown emphasis around the candidate.
    s = re.sub(r"^\*+|\*+$", "", s).strip()

    # Remove trailing prose punctuation only.
    s = s.rstrip().rstrip(".")

    return s.strip()


def extract_math500_answer(generated_text, was_truncated=False, finish_reason=None):
    """
    Conservative final-answer extractor.

    Returns:
        extracted_answer, extraction_method
    """
    if not isinstance(generated_text, str) or not generated_text.strip():
        return "", "empty_generation"

    text = generated_text.strip()

    # DeepSeek reasoning/final-response boundary.
    if "</think>" in text:
        post_think = text.rsplit("</think>", 1)[-1].strip()
    else:
        post_think = text

    # --------------------------------------------------------------
    # 1. Exact format originally requested by the benchmark
    # --------------------------------------------------------------
    matches = re.findall(
        r"FINAL_ANSWER\s*:\s*([^\r\n]+)",
        post_think,
        flags=re.I,
    )

    if matches:
        return clean_candidate(matches[-1]), "explicit_FINAL_ANSWER"

    # --------------------------------------------------------------
    # 2. Boxed answer in final-response region
    # --------------------------------------------------------------
    boxes = extract_balanced_command(post_think, r"\boxed")

    if boxes:
        return clean_candidate(boxes[-1]), "boxed_post_think"

    # --------------------------------------------------------------
    # 3. Explicit "Final Answer" marker
    # --------------------------------------------------------------
    markers = list(
        re.finditer(
            r"(?:\*\*)?\s*Final\s+Answer\s*(?:\*\*)?\s*:?",
            post_think,
            flags=re.I,
        )
    )

    if markers:
        tail = post_think[markers[-1].end():].strip()

        boxes = extract_balanced_command(tail, r"\boxed")
        if boxes:
            return clean_candidate(boxes[-1]), "boxed_after_final_marker"

        for line in tail.splitlines():
            candidate = line.strip().strip("$").strip()

            candidate = (
                candidate
                .replace(r"\[", "")
                .replace(r"\]", "")
                .strip()
            )

            candidate = re.sub(
                r"^[-*#>\s]+",
                "",
                candidate,
            ).strip()

            if not candidate:
                continue

            m = re.search(
                r"(?:the\s+)?(?:final\s+)?answer\s+(?:is|=)\s*(.+?)[.!]?$",
                candidate,
                flags=re.I,
            )

            if m:
                return clean_candidate(m.group(1)), "final_marker_natural"

            # Keep only reasonably short direct-answer lines.
            if len(candidate) <= 200:
                return clean_candidate(candidate), "after_final_marker"

    # --------------------------------------------------------------
    # 4. Last boxed answer in the complete generation
    # --------------------------------------------------------------
    boxes = extract_balanced_command(text, r"\boxed")

    if boxes:
        return clean_candidate(boxes[-1]), "boxed_full_text"

    # --------------------------------------------------------------
    # 5. Conservative natural-language answer in final response only
    # --------------------------------------------------------------
    patterns = [
        r"(?:the\s+)?final\s+answer\s+(?:is|=)\s*(.+?)(?:\n|$)",
        r"(?:therefore|thus|hence),?\s+(?:the\s+)?answer\s+(?:is|=)\s*(.+?)(?:\n|$)",
    ]

    for pattern in patterns:
        values = re.findall(pattern, post_think, flags=re.I)

        if values:
            return (
                clean_candidate(values[-1]),
                "natural_language_post_think",
            )

    # --------------------------------------------------------------
    # Do NOT infer an answer from unfinished reasoning.
    # --------------------------------------------------------------
    if bool(was_truncated) or str(finish_reason).lower() == "length":
        return "", "truncated_no_final_answer"

    return "", "unparseable"

## Normalize and compare mathematical answers

The scorer first checks normalized exact equality and then tries symbolic equivalence with SymPy.

This helps match equivalent forms such as:

```text
\dfrac{33}{100}  ==  \frac{33}{100}
1 / sqrt(3)       ==  sqrt(3) / 3
106^\circ         ==  106°
```

If symbolic parsing cannot safely handle an expression, the notebook falls back to normalized-string comparison rather than guessing.


In [8]:
try:
    from sympy.parsing.latex import parse_latex
    LATEX_PARSER_AVAILABLE = True
except Exception:
    LATEX_PARSER_AVAILABLE = False

print("SymPy LaTeX parser available:", LATEX_PARSER_AVAILABLE)


def normalize_math_string(value):
    if value is None or (isinstance(value, float) and np.isnan(value)):
        return ""

    s = str(value).strip()

    # Markdown / math delimiters
    s = s.replace("$", "")
    s = s.replace(r"\[", "").replace(r"\]", "")
    s = s.replace(r"\(", "").replace(r"\)", "")

    # LaTeX visual-only commands
    s = s.replace(r"\left", "").replace(r"\right", "")
    s = s.replace(r"\!", "").replace(r"\,", "")
    s = s.replace(r"\;", "").replace(r"\:", "")

    # Normalize fraction commands
    s = s.replace(r"\dfrac", r"\frac")
    s = s.replace(r"\tfrac", r"\frac")

    # Degrees
    s = s.replace(r"^\circ", "°")
    s = s.replace(r"^{\circ}", "°")
    s = s.replace(r"\degree", "°")

    # Text-only categorical answers such as \\text{east}
    s = re.sub(
        r"\\(?:text|mathrm|operatorname)\s*\{([^{}]*)\}",
        r"\1",
        s,
    )

    # Common formatting
    s = s.replace("−", "-")
    s = s.replace("–", "-")
    s = s.replace("×", r"\times")
    s = re.sub(r"\s+", "", s)

    # Normalize simple coordinate spacing / commas
    s = s.replace(",\\ ", ",")

    # Case-insensitive for textual answers.
    return s.strip().lower()


def strip_outer_boxed(s):
    boxes = extract_balanced_command(str(s), r"\boxed")
    if boxes:
        return boxes[-1]
    return str(s)


def try_parse_latex_expr(s):
    """
    Parse a scalar/algebraic LaTeX expression with SymPy.
    Returns None when parsing is unsuitable.
    """
    if not LATEX_PARSER_AVAILABLE:
        return None

    s = strip_outer_boxed(s).strip()

    # SymPy LaTeX parser is not ideal for these structures.
    if not s:
        return None

    # Categorical text
    if re.search(r"\\text\s*\{", s):
        return None

    # Coordinates / tuples are compared separately.
    if (
        s.startswith("(")
        and s.endswith(")")
        and "," in s
    ):
        return None

    # Degree measures: compare numeric part separately.
    s = (
        s.replace(r"^\circ", "")
         .replace(r"^{\circ}", "")
         .replace("°", "")
    )

    try:
        return parse_latex(s)
    except Exception:
        return None


def parse_numeric_string(s):
    s = normalize_math_string(s)

    # Remove degree marker for numeric comparison.
    s = s.replace("°", "")

    # Plain integer/decimal
    try:
        return float(s)
    except Exception:
        return None


def split_top_level_tuple(s):
    s = normalize_math_string(s)

    if not (
        len(s) >= 3
        and s[0] in "(["
        and s[-1] in ")]"
        and "," in s
    ):
        return None

    inner = s[1:-1]

    # Math500 coordinates in this benchmark are simple two-element tuples.
    parts = inner.split(",")

    if len(parts) != 2:
        return None

    return parts[0], parts[1]


def math500_equal(predicted, gold):
    pred_raw = strip_outer_boxed(predicted)
    gold_raw = strip_outer_boxed(gold)

    p = normalize_math_string(pred_raw)
    g = normalize_math_string(gold_raw)

    if not p or not g:
        return False, "missing_answer"

    # --------------------------------------------------------------
    # 1. Exact normalized equality
    # --------------------------------------------------------------
    if p == g:
        return True, "normalized_exact"

    # --------------------------------------------------------------
    # 2. Compare simple tuples / coordinates component-wise
    # --------------------------------------------------------------
    p_tuple = split_top_level_tuple(p)
    g_tuple = split_top_level_tuple(g)

    if p_tuple is not None and g_tuple is not None:
        flags = []

        for pp, gg in zip(p_tuple, g_tuple):
            ok, _ = math500_equal(pp, gg)
            flags.append(ok)

        return all(flags), "tuple_component_compare"

    # --------------------------------------------------------------
    # 3. Plain numeric comparison
    # --------------------------------------------------------------
    p_num = parse_numeric_string(p)
    g_num = parse_numeric_string(g)

    if p_num is not None and g_num is not None:
        return (
            math.isclose(
                p_num,
                g_num,
                rel_tol=1e-9,
                abs_tol=1e-9,
            ),
            "numeric_compare",
        )

    # --------------------------------------------------------------
    # 4. Symbolic LaTeX equivalence
    # --------------------------------------------------------------
    p_expr = try_parse_latex_expr(pred_raw)
    g_expr = try_parse_latex_expr(gold_raw)

    if p_expr is not None and g_expr is not None:
        try:
            diff = sp.simplify(p_expr - g_expr)

            if diff == 0:
                return True, "sympy_equivalent"

            equals = p_expr.equals(g_expr)

            if equals is True:
                return True, "sympy_equivalent"

            return False, "sympy_not_equal"

        except Exception:
            pass

    # --------------------------------------------------------------
    # 5. Textual fallback
    # --------------------------------------------------------------
    p_text = re.sub(r"[^a-z0-9_+\-*/().,=]", "", p)
    g_text = re.sub(r"[^a-z0-9_+\-*/().,=]", "", g)

    if p_text and p_text == g_text:
        return True, "text_fallback"

    return False, "not_equivalent"

SymPy LaTeX parser available: True


## Repair and rescore every Math500 row

In [10]:
repaired = df.copy()

# Preserve the original evaluation columns for auditability.
for col in [
    "final_answer",
    "parse_status",
    "parse_success",
    "correct",
    "evaluation_method",
]:
    if col in repaired.columns:
        repaired[f"original_{col}"] = repaired[col]

# New repair-specific columns.
repaired["repair_extracted_answer"] = ""
repaired["repair_extraction_method"] = ""
repaired["repair_comparison_method"] = ""
repaired["repair_changed_parse"] = False
repaired["repair_changed_correctness"] = False

math_mask = repaired["dataset"].astype(str).str.lower().eq("math500")

for idx in repaired.index[math_mask]:
    row = repaired.loc[idx]

    extracted, extraction_method = extract_math500_answer(
        row["generated_text"],
        was_truncated=row.get("was_truncated", False),
        finish_reason=row.get("finish_reason", None),
    )

    repaired.at[idx, "repair_extracted_answer"] = extracted
    repaired.at[idx, "repair_extraction_method"] = extraction_method

    if extracted:
        is_correct, comparison_method = math500_equal(
            extracted,
            row["gold_answer"],
        )

        repaired.at[idx, "repair_comparison_method"] = comparison_method
        repaired.at[idx, "final_answer"] = extracted
        repaired.at[idx, "parse_status"] = (
            "repaired_final_answer_found"
        )
        repaired.at[idx, "parse_success"] = True
        repaired.at[idx, "correct"] = bool(is_correct)
        repaired.at[idx, "evaluation_method"] = (
            f"math500_repaired::{comparison_method}"
        )

    else:
        repaired.at[idx, "repair_comparison_method"] = "not_scored"

        if extraction_method == "truncated_no_final_answer":
            repaired.at[idx, "parse_status"] = (
                "truncated_no_final_answer"
            )
        else:
            repaired.at[idx, "parse_status"] = (
                "repair_unparseable"
            )

        repaired.at[idx, "parse_success"] = False
        repaired.at[idx, "correct"] = False
        repaired.at[idx, "evaluation_method"] = (
            "math500_repair_failed"
        )

# Change tracking.
repaired.loc[math_mask, "repair_changed_parse"] = (
    repaired.loc[math_mask, "parse_success"].astype(bool).values
    != repaired.loc[math_mask, "original_parse_success"].astype(bool).values
)

repaired.loc[math_mask, "repair_changed_correctness"] = (
    repaired.loc[math_mask, "correct"].astype(bool).values
    != repaired.loc[math_mask, "original_correct"].astype(bool).values
)

print("Math500 repair complete.")

Math500 repair complete.


## Check extraction coverage and corrected accuracy

In [11]:
math_after = repaired[math_mask].copy()

coverage = (
    math_after.groupby("precision")
    .agg(
        rows=("sample_id", "size"),
        parsed=("parse_success", "sum"),
        correct=("correct", "sum"),
        truncated=("was_truncated", "sum"),
    )
)

coverage["parse_success_pct"] = (
    coverage["parsed"] / coverage["rows"] * 100
)

coverage["accuracy_pct"] = (
    coverage["correct"] / coverage["rows"] * 100
)

display(coverage.round(2))

print("\nExtraction methods:")
display(
    math_after["repair_extraction_method"]
    .value_counts(dropna=False)
    .rename_axis("method")
    .reset_index(name="rows")
)

print("\nComparison methods:")
display(
    math_after["repair_comparison_method"]
    .value_counts(dropna=False)
    .rename_axis("method")
    .reset_index(name="rows")
)

,rows,parsed,correct,truncated,parse_success_pct,accuracy_pct
precision,,,,,,
BF16,50,46,43,4,92.0,86.0
Q3_K_M,50,47,45,4,94.0,90.0
Q4_K_M,50,47,46,3,94.0,92.0
Q5_K_M,50,47,45,3,94.0,90.0
Q6_K,50,48,46,3,96.0,92.0
Q8_0,50,47,44,3,94.0,88.0



Extraction methods:


,method,rows
0,boxed_post_think,272
1,truncated_no_final_answer,18
2,explicit_FINAL_ANSWER,7
3,boxed_full_text,2
4,after_final_marker,1



Comparison methods:


,method,rows
0,normalized_exact,243
1,not_scored,18
2,sympy_equivalent,13
3,sympy_not_equal,12
4,numeric_compare,9
5,tuple_component_compare,3
6,text_fallback,2


## Inspect unresolved rows

These should mostly be generations that hit the token limit before producing a final answer.

The notebook intentionally leaves these unresolved instead of extracting a speculative answer from chain-of-thought.


In [12]:
unresolved = math_after[~math_after["parse_success"].astype(bool)].copy()

print(f"Unresolved Math500 rows: {len(unresolved)}")

display(
    unresolved[
        [
            "precision",
            "sample_id",
            "gold_answer",
            "finish_reason",
            "was_truncated",
            "repair_extraction_method",
            "parse_status",
        ]
    ].sort_values(["sample_id", "precision"])
)


Unresolved Math500 rows: 18


,precision,sample_id,gold_answer,finish_reason,was_truncated,repair_extraction_method,parse_status
104,BF16,Q007,16,length,True,truncated_no_final_answer,truncated_no_final_answer
1484,Q3_K_M,Q007,16,length,True,truncated_no_final_answer,truncated_no_final_answer
1183,Q4_K_M,Q037,\frac{14}{3},length,True,truncated_no_final_answer,truncated_no_final_answer
102,BF16,Q038,1+274i,length,True,truncated_no_final_answer,truncated_no_final_answer
1206,Q4_K_M,Q038,1+274i,length,True,truncated_no_final_answer,truncated_no_final_answer
930,Q5_K_M,Q038,1+274i,length,True,truncated_no_final_answer,truncated_no_final_answer
654,Q6_K,Q038,1+274i,length,True,truncated_no_final_answer,truncated_no_final_answer
1421,Q3_K_M,Q052,1,length,True,truncated_no_final_answer,truncated_no_final_answer
262,BF16,Q061,\text{even},length,True,truncated_no_final_answer,truncated_no_final_answer
336,Q8_0,Q074,180^\circ,length,True,truncated_no_final_answer,truncated_no_final_answer


## Audit newly recovered answers

In [13]:
recovered = math_after[
    (~math_after["original_parse_success"].astype(bool))
    & (math_after["parse_success"].astype(bool))
].copy()

print(f"Newly recovered Math500 answers: {len(recovered)}")

display(
    recovered[
        [
            "precision",
            "sample_id",
            "gold_answer",
            "repair_extracted_answer",
            "repair_extraction_method",
            "repair_comparison_method",
            "correct",
        ]
    ].head(40)
)

Newly recovered Math500 answers: 275


,precision,sample_id,gold_answer,repair_extracted_answer,repair_extraction_method,repair_comparison_method,correct
1,BF16,Q091,\dfrac{33}{100},\dfrac{33}{100},boxed_post_think,normalized_exact,True
3,BF16,Q071,2,2,boxed_post_think,normalized_exact,True
6,BF16,Q065,\cot x,\cot x,boxed_post_think,normalized_exact,True
9,BF16,Q096,"(-1,6)","(-1,\ 6)",boxed_post_think,tuple_component_compare,True
11,BF16,Q014,\frac{\sqrt{3}}{3},\dfrac{\sqrt{3}}{3},boxed_post_think,normalized_exact,True
15,BF16,Q008,23,23,boxed_post_think,normalized_exact,True
26,BF16,Q089,10,10,boxed_post_think,normalized_exact,True
29,BF16,Q005,11,11,boxed_post_think,normalized_exact,True
31,BF16,Q009,21,21,boxed_post_think,normalized_exact,True
33,BF16,Q073,210,210,boxed_post_think,normalized_exact,True


## Save corrected results to new files

The original CSV remains unchanged.

The notebook writes:

- `all_results_math500_rescored.csv` — complete benchmark with repaired Math500 rows
- `math500_rescore_audit.csv` — Math500-only audit data
- `math500_rescore_summary.csv` — per-precision summary


In [15]:
summary = coverage.reset_index()

audit_columns = [
    "sample_id",
    "precision",
    "question",
    "gold_answer",
    "generated_text",
    "finish_reason",
    "was_truncated",
    "original_final_answer",
    "original_parse_status",
    "original_parse_success",
    "original_correct",
    "repair_extracted_answer",
    "repair_extraction_method",
    "repair_comparison_method",
    "final_answer",
    "parse_status",
    "parse_success",
    "correct",
    "repair_changed_parse",
    "repair_changed_correctness",
]

audit_columns = [
    c for c in audit_columns
    if c in math_after.columns
]

repaired.to_csv(
    CORRECTED_RESULTS_PATH,
    index=False,
)

math_after[audit_columns].to_csv(
    MATH500_AUDIT_PATH,
    index=False,
)

summary.to_csv(
    SUMMARY_PATH,
    index=False,
)

print("Saved:")
print("  Corrected full results:", CORRECTED_RESULTS_PATH)
print("  Math500 audit         :", MATH500_AUDIT_PATH)
print("  Math500 summary       :", SUMMARY_PATH)

Saved:
  Corrected full results: /content/drive/MyDrive/DeepSeek_R1_7B_Quantization_Benchmark_v2/rescored/math500_parse_repair/all_results_math500_rescored.csv
  Math500 audit         : /content/drive/MyDrive/DeepSeek_R1_7B_Quantization_Benchmark_v2/rescored/math500_parse_repair/math500_rescore_audit.csv
  Math500 summary       : /content/drive/MyDrive/DeepSeek_R1_7B_Quantization_Benchmark_v2/rescored/math500_parse_repair/math500_rescore_summary.csv


## Before vs after comparison

In [16]:
before = (
    df[df["dataset"].astype(str).str.lower().eq("math500")]
    .groupby("precision")
    .agg(
        original_parse_success=("parse_success", "mean"),
        original_accuracy=("correct", "mean"),
    )
)

after = (
    repaired[math_mask]
    .groupby("precision")
    .agg(
        repaired_parse_success=("parse_success", "mean"),
        repaired_accuracy=("correct", "mean"),
    )
)

comparison = before.join(after)

for c in comparison.columns:
    comparison[c] = comparison[c] * 100

comparison["parse_gain_pp"] = (
    comparison["repaired_parse_success"]
    - comparison["original_parse_success"]
)

comparison["accuracy_gain_pp"] = (
    comparison["repaired_accuracy"]
    - comparison["original_accuracy"]
)

display(comparison.round(2))

,original_parse_success,original_accuracy,repaired_parse_success,repaired_accuracy,parse_gain_pp,accuracy_gain_pp
precision,,,,,,
BF16,2.0,0.0,92.0,86.0,90.0,86.0
Q3_K_M,4.0,4.0,94.0,90.0,90.0,86.0
Q4_K_M,6.0,4.0,94.0,92.0,88.0,88.0
Q5_K_M,0.0,0.0,94.0,90.0,94.0,90.0
Q6_K,2.0,0.0,96.0,92.0,94.0,92.0
Q8_0,0.0,0.0,94.0,88.0,94.0,88.0


## Recompute overall benchmark accuracy

This updates the aggregate accuracy after Math500 repair while leaving all other dataset evaluations unchanged.


In [17]:
overall_before = (
    df.groupby("precision")["correct"]
    .mean()
    .mul(100)
    .rename("accuracy_before_pct")
)

overall_after = (
    repaired.groupby("precision")["correct"]
    .mean()
    .mul(100)
    .rename("accuracy_after_pct")
)

overall = pd.concat(
    [overall_before, overall_after],
    axis=1,
)

overall["change_pp"] = (
    overall["accuracy_after_pct"]
    - overall["accuracy_before_pct"]
)

display(overall.round(2))


,accuracy_before_pct,accuracy_after_pct,change_pp
precision,,,
BF16,37.6,54.8,17.2
Q3_K_M,35.6,52.8,17.2
Q4_K_M,34.8,52.4,17.6
Q5_K_M,37.6,55.6,18.0
Q6_K,35.6,54.0,18.4
Q8_0,35.2,52.8,17.6


## Optional: exact McNemar comparison against BF16 after repair

Use this only after checking the audit file and confirming that the repaired Math500 parsing is satisfactory.


In [18]:
from math import comb

def exact_two_sided_binom_pvalue(k, n):
    if n == 0:
        return 1.0

    # Exact two-sided sign/binomial test for p=0.5.
    probs = [
        comb(n, i) * (0.5 ** n)
        for i in range(n + 1)
    ]

    observed = probs[k]

    return min(
        1.0,
        sum(p for p in probs if p <= observed + 1e-15),
    )


def paired_correctness_table(results, baseline="BF16"):
    wide = (
        results.pivot_table(
            index="sample_id",
            columns="precision",
            values="correct",
            aggfunc="first",
        )
    )

    rows = []

    for precision in wide.columns:
        if precision == baseline:
            continue

        pair = wide[[baseline, precision]].dropna()

        b = int(
            ((pair[baseline] == True) & (pair[precision] == False)).sum()
        )

        c = int(
            ((pair[baseline] == False) & (pair[precision] == True)).sum()
        )

        rows.append(
            {
                "precision": precision,
                "BF16_correct_quant_wrong": b,
                "BF16_wrong_quant_correct": c,
                "discordant_pairs": b + c,
                "exact_p_value": exact_two_sided_binom_pvalue(
                    min(b, c),
                    b + c,
                ),
            }
        )

    return pd.DataFrame(rows)


mcnemar_after = paired_correctness_table(repaired)

display(mcnemar_after)


,precision,BF16_correct_quant_wrong,BF16_wrong_quant_correct,discordant_pairs,exact_p_value
0,Q3_K_M,23,18,41,0.532709
1,Q4_K_M,24,18,42,0.440799
2,Q5_K_M,22,24,46,0.882996
3,Q6_K,24,22,46,0.882996
4,Q8_0,30,25,55,0.590053
